# Exploratory Data Analysis: English and Romanian Narrative Similarity Datasets

- individual EDA for English files;
- individual EDA for Romanian files;
- common/aligned English-Romanian analysis;
- label distributions for Track A-style triplets;
- story-length, token-length, overlap, duplicate, and coverage checks;

## 1. Setup

In [ ]:
import json
import math
import os
import re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    sns = None
    plt.style.use("default")

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)

SEED = int(os.environ.get("REPRODUCIBLE_SEED", 42))
np.random.seed(SEED)

print("Ready. Seed:", SEED)

In [ ]:
EN_DATA_DIR = Path(os.environ.get("EN_DATA_DIR", "/kaggle/input/datasets/anisoaraana/similarity"))
RO_DATA_DIR = Path(os.environ.get("RO_DATA_DIR", "/kaggle/input/datasets/anisoaraana/romanian-narrative-similarity-dataset"))
RO_SUFFIX = os.environ.get("RO_SUFFIX", "_ro")

TRANSLATION_CACHE_PATH = Path(os.environ.get("TRANSLATION_CACHE_PATH", RO_DATA_DIR / "translation_cache_en_ro.json"))
ASPECTS_CACHE_PATH = Path(os.environ.get("ASPECTS_CACHE_PATH", EN_DATA_DIR / "aspects_cache.json"))

print("English data dir:", EN_DATA_DIR)
print("Romanian data dir:", RO_DATA_DIR)
print("Romanian suffix:", RO_SUFFIX)
print("Translation cache:", TRANSLATION_CACHE_PATH)
print("English aspects cache:", ASPECTS_CACHE_PATH)

## 3. Utility Functions

In [ ]:
def first_existing(paths):
    for path in paths:
        if Path(path).exists():
            return Path(path)
    return Path(paths[0])


def ro_file(stem):
    return first_existing([
        RO_DATA_DIR / f"{stem}{RO_SUFFIX}.jsonl",
        RO_DATA_DIR / f"{stem}.jsonl",
        EN_DATA_DIR / f"{stem}{RO_SUFFIX}.jsonl",
    ])


EN_PATHS = {
    "synthetic": EN_DATA_DIR / "synthetic_data_for_classification.jsonl",
    "extra_synthetic": EN_DATA_DIR / "synthetic_data_new.jsonl",
    "extra_tell_me_again": EN_DATA_DIR / "tell_me_again_triplets.jsonl",
    "dev_a": EN_DATA_DIR / "dev_track_a.jsonl",
    "test_a": EN_DATA_DIR / "test_track_a.jsonl",
    "test_b": EN_DATA_DIR / "test_track_b.jsonl",
    "test_a_labels": EN_DATA_DIR / "test_track_a_labels.jsonl",
    "test_b_labels": EN_DATA_DIR / "test_track_b_labels.jsonl",
}

RO_PATHS = {
    "synthetic": ro_file("synthetic_data_for_classification"),
    "extra_synthetic": ro_file("synthetic_data_new"),
    "dev_a": ro_file("dev_track_a"),
    "test_a": ro_file("test_track_a"),
    "test_b": ro_file("test_track_b"),
}

def path_status_table(paths, language):
    rows = []
    for name, path in paths.items():
        rows.append({
            "language": language,
            "file": name,
            "path": str(path),
            "exists": path.exists(),
            "size_mb": path.stat().st_size / (1024 ** 2) if path.exists() else np.nan,
        })
    return pd.DataFrame(rows)

pd.concat([
    path_status_table(EN_PATHS, "English"),
    path_status_table(RO_PATHS, "Romanian"),
], ignore_index=True)

In [12]:
def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        print(f"Missing file: {path}")
        return pd.DataFrame()
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


def norm_text(text):
    return " ".join(str(text).split())


def simple_word_count(text):
    return len(re.findall(r"\b\w+\b", str(text), flags=re.UNICODE))


def sentence_count(text):
    parts = re.split(r"[.!?]+", str(text))
    return sum(1 for p in parts if p.strip())


def add_text_stats(df, text_columns):
    out = df.copy()
    for col in text_columns:
        if col in out.columns:
            out[f"{col}_chars"] = out[col].astype(str).map(len)
            out[f"{col}_words"] = out[col].astype(str).map(simple_word_count)
            out[f"{col}_sentences"] = out[col].astype(str).map(sentence_count)
            out[f"{col}_norm"] = out[col].astype(str).map(norm_text)
    return out


def summarize_numeric(series):
    s = pd.Series(series).dropna()
    if len(s) == 0:
        return {"n": 0}
    return {
        "n": int(len(s)),
        "mean": float(s.mean()),
        "std": float(s.std(ddof=0)),
        "min": float(s.min()),
        "p25": float(s.quantile(0.25)),
        "median": float(s.median()),
        "p75": float(s.quantile(0.75)),
        "max": float(s.max()),
    }


def text_columns_for_df(df):
    candidates = ["anchor_text", "text_a", "text_b", "text", "anchor", "positive", "negative"]
    return [c for c in candidates if c in df.columns]


def plot_hist(series, title, xlabel, bins=40):
    s = pd.Series(series).dropna()
    if len(s) == 0:
        print("No data for", title)
        return
    plt.figure(figsize=(8, 4))
    if sns is not None:
        sns.histplot(s, bins=bins, kde=False)
    else:
        plt.hist(s, bins=bins)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()


def plot_box_by_dataset(summary_df, value_col, title):
    if summary_df.empty or value_col not in summary_df.columns:
        print("No data for", title)
        return
    plt.figure(figsize=(10, 4))
    if sns is not None:
        sns.boxplot(data=summary_df, x="dataset", y=value_col, hue="language")
    else:
        summary_df.boxplot(column=value_col, by=["dataset", "language"], rot=45)
    plt.title(title)
    plt.xlabel("Dataset")
    plt.ylabel(value_col)
    plt.tight_layout()
    plt.show()

## 4. Load Datasets

In [ ]:
en = {name: read_jsonl(path) for name, path in EN_PATHS.items()}
ro = {name: read_jsonl(path) for name, path in RO_PATHS.items()}

shape_rows = []
for language, data in [("English", en), ("Romanian", ro)]:
    for name, df in data.items():
        shape_rows.append({
            "language": language,
            "dataset": name,
            "rows": len(df),
            "columns": ", ".join(df.columns.tolist()) if len(df.columns) else "",
        })
shape_df = pd.DataFrame(shape_rows)
shape_df

## 5. Individual Analysis: Dataset Schemas and Missing Values

In [ ]:
def _safe_unique_value(value):
    if isinstance(value, (list, dict, tuple, set)):
        return json.dumps(value, sort_keys=True, ensure_ascii=False, default=str)
    return value

def safe_nunique(series):
    # Some JSONL columns may contain lists/dicts. pandas.nunique cannot hash them directly.
    normalized = series.map(_safe_unique_value)
    return int(normalized.dropna().nunique())

def schema_report(data, language):
    rows = []
    for name, df in data.items():
        for col in df.columns:
            rows.append({
                "language": language,
                "dataset": name,
                "column": col,
                "dtype": str(df[col].dtype),
                "missing": int(df[col].isna().sum()),
                "missing_pct": float(df[col].isna().mean() * 100) if len(df) else 0.0,
                "unique": safe_nunique(df[col]),
            })
    return pd.DataFrame(rows)

schema = pd.concat([
    schema_report(en, "English"),
    schema_report(ro, "Romanian"),
], ignore_index=True)
schema.sort_values(["language", "dataset", "column"])

## 6. Individual Analysis: Label Distributions

In [ ]:
def label_distribution(data, language):
    rows = []
    for name, df in data.items():
        if "text_a_is_closer" not in df.columns:
            continue
        labels = df["text_a_is_closer"].astype(bool)
        counts = labels.value_counts(dropna=False).to_dict()
        rows.append({
            "language": language,
            "dataset": name,
            "n": len(labels),
            "true_count": int(counts.get(True, 0)),
            "false_count": int(counts.get(False, 0)),
            "true_pct": float(labels.mean() * 100) if len(labels) else 0.0,
        })
    return pd.DataFrame(rows)

labels_df = pd.concat([
    label_distribution(en, "English"),
    label_distribution(ro, "Romanian"),
], ignore_index=True)
labels_df

In [ ]:
if not labels_df.empty:
    plt.figure(figsize=(9, 4))
    if sns is not None:
        sns.barplot(data=labels_df, x="dataset", y="true_pct", hue="language")
    else:
        labels_df.pivot(index="dataset", columns="language", values="true_pct").plot(kind="bar")
    plt.axhline(50, color="gray", linestyle="--", linewidth=1)
    plt.title("Percentage of examples where text_a_is_closer is True")
    plt.ylabel("True label (%)")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

## 7. Individual Analysis: Text Length Statistics

In [ ]:
def long_text_stats(data, language):
    rows = []
    long_rows = []
    for name, df in data.items():
        cols = text_columns_for_df(df)
        df_stats = add_text_stats(df, cols)
        for col in cols:
            for unit in ["chars", "words", "sentences"]:
                stat_col = f"{col}_{unit}"
                s = summarize_numeric(df_stats[stat_col])
                s.update({"language": language, "dataset": name, "text_column": col, "unit": unit})
                rows.append(s)
            tmp = df_stats[[f"{col}_chars", f"{col}_words", f"{col}_sentences"]].copy()
            tmp["language"] = language
            tmp["dataset"] = name
            tmp["text_column"] = col
            tmp = tmp.rename(columns={f"{col}_chars": "chars", f"{col}_words": "words", f"{col}_sentences": "sentences"})
            long_rows.append(tmp)
    return pd.DataFrame(rows), pd.concat(long_rows, ignore_index=True) if long_rows else pd.DataFrame()

length_summary_en, length_long_en = long_text_stats(en, "English")
length_summary_ro, length_long_ro = long_text_stats(ro, "Romanian")
length_summary = pd.concat([length_summary_en, length_summary_ro], ignore_index=True)
length_long = pd.concat([length_long_en, length_long_ro], ignore_index=True)

length_summary.sort_values(["language", "dataset", "text_column", "unit"]).head(80)

In [ ]:
plot_box_by_dataset(length_long, "words", "Word-count distribution by dataset and language")

In [ ]:
for language, data in [("English", en), ("Romanian", ro)]:
    df = data.get("test_b", pd.DataFrame())
    if not df.empty and "text" in df.columns:
        stats = add_text_stats(df, ["text"])
        plot_hist(stats["text_words"], f"{language} Track B story word counts", "Words per story")

## 8. Individual Analysis: Duplicates and Text Reuse

In [ ]:
def duplicate_report(data, language):
    rows = []
    for name, df in data.items():
        cols = text_columns_for_df(df)
        for col in cols:
            norm = df[col].astype(str).map(norm_text)
            rows.append({
                "language": language,
                "dataset": name,
                "text_column": col,
                "rows": len(norm),
                "unique_texts": int(norm.nunique()),
                "duplicate_rows": int(len(norm) - norm.nunique()),
                "duplicate_pct": float((len(norm) - norm.nunique()) / len(norm) * 100) if len(norm) else 0.0,
            })
    return pd.DataFrame(rows)

dups = pd.concat([
    duplicate_report(en, "English"),
    duplicate_report(ro, "Romanian"),
], ignore_index=True)
dups.sort_values(["language", "dataset", "text_column"])

In [ ]:
def collect_all_texts(data):
    counter = Counter()
    provenance = defaultdict(list)
    for name, df in data.items():
        for col in text_columns_for_df(df):
            for text in df[col].astype(str):
                key = norm_text(text)
                counter[key] += 1
                provenance[key].append(f"{name}:{col}")
    return counter, provenance

for language, data in [("English", en), ("Romanian", ro)]:
    counter, provenance = collect_all_texts(data)
    repeated = [(text, count) for text, count in counter.items() if count > 1]
    print(f"{language}: total unique texts={len(counter)}, repeated texts={len(repeated)}")
    display(pd.DataFrame(repeated, columns=["text", "count"]).sort_values("count", ascending=False).head(10))

## 9. English Aspect Cache Diagnostics

In [ ]:
def read_json_or_jsonl(path):
    path = Path(path)
    if not path.exists():
        return None
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        return None
    if text[0] in "[{":
        try:
            return json.loads(text)
        except Exception:
            pass
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def load_aspects_cache(path):
    raw = read_json_or_jsonl(path)
    cache = {}
    if raw is None:
        return cache
    def add(story_text, payload):
        if not story_text or not isinstance(payload, dict):
            return
        cache[norm_text(story_text)] = {
            "coa": payload.get("coa") or payload.get("course_of_action") or payload.get("plot") or payload.get("action") or "",
            "outcomes": payload.get("outcomes") or payload.get("outcome") or payload.get("ending") or "",
            "theme": payload.get("theme") or payload.get("abstract_theme") or "",
        }
    if isinstance(raw, list):
        for obj in raw:
            if isinstance(obj, dict):
                add(obj.get("text") or obj.get("raw_text") or obj.get("story"), obj)
    elif isinstance(raw, dict):
        if isinstance(raw.get("records"), list):
            for obj in raw["records"]:
                add(obj.get("text") or obj.get("raw_text") or obj.get("story"), obj)
        else:
            for k, v in raw.items():
                add(k, v)
    return cache


aspects_cache = load_aspects_cache(ASPECTS_CACHE_PATH)
print("Aspect cache entries:", len(aspects_cache))

In [ ]:
def aspect_coverage_for_data(data, language="English"):
    rows = []
    for name, df in data.items():
        cols = text_columns_for_df(df)
        texts = []
        for col in cols:
            texts.extend(df[col].astype(str).map(norm_text).tolist())
        unique_texts = sorted(set(texts))
        for aspect in ["coa", "outcomes", "theme"]:
            hits = sum(1 for t in unique_texts if t in aspects_cache and str(aspects_cache[t].get(aspect, "")).strip())
            rows.append({
                "language": language,
                "dataset": name,
                "aspect": aspect,
                "unique_texts": len(unique_texts),
                "hits": hits,
                "hit_rate": hits / len(unique_texts) if unique_texts else 0.0,
            })
    return pd.DataFrame(rows)

aspect_cov = aspect_coverage_for_data(en, "English") if aspects_cache else pd.DataFrame()
aspect_cov

In [ ]:
if not aspect_cov.empty:
    plt.figure(figsize=(10, 4))
    if sns is not None:
        sns.barplot(data=aspect_cov, x="dataset", y="hit_rate", hue="aspect")
    else:
        aspect_cov.pivot(index="dataset", columns="aspect", values="hit_rate").plot(kind="bar")
    plt.title("English aspect cache coverage")
    plt.ylabel("Hit rate")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

## 10. Common Analysis: English-Romanian Alignment

In [ ]:
def load_translation_cache(path):
    path = Path(path)
    if not path.exists():
        print("Missing translation cache:", path)
        return {}
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)
    if not isinstance(raw, dict):
        print("Translation cache is not a JSON object")
        return {}
    return {norm_text(k): norm_text(v) for k, v in raw.items()}

translation_cache = load_translation_cache(TRANSLATION_CACHE_PATH)
print("Translation cache entries:", len(translation_cache))

In [ ]:
def unique_texts_from_data(data):
    texts = set()
    for name, df in data.items():
        for col in text_columns_for_df(df):
            texts.update(df[col].astype(str).map(norm_text).tolist())
    return texts

en_unique_texts = unique_texts_from_data({k: v for k, v in en.items() if k not in ["test_a_labels", "test_b_labels"]})
cache_keys = set(translation_cache.keys())
cache_values = set(translation_cache.values())

coverage = {
    "english_unique_texts": len(en_unique_texts),
    "translation_cache_entries": len(translation_cache),
    "english_texts_found_in_cache": len(en_unique_texts & cache_keys),
    "coverage_pct": len(en_unique_texts & cache_keys) / len(en_unique_texts) * 100 if en_unique_texts else 0.0,
    "unique_romanian_translations": len(cache_values),
}
pd.DataFrame([coverage])

In [ ]:
def alignment_report(en_data, ro_data):
    rows = []
    for name in sorted(set(en_data) & set(ro_data)):
        en_df = en_data[name]
        ro_df = ro_data[name]
        rows.append({
            "dataset": name,
            "english_rows": len(en_df),
            "romanian_rows": len(ro_df),
            "same_row_count": len(en_df) == len(ro_df),
            "english_columns": ", ".join(en_df.columns),
            "romanian_columns": ", ".join(ro_df.columns),
        })
    return pd.DataFrame(rows)

align_rows = alignment_report(en, ro)
align_rows

## 11. Common Analysis: English-Romanian Length Ratios

In [ ]:
def paired_length_rows(dataset_name, en_df, ro_df):
    rows = []
    if en_df.empty or ro_df.empty or len(en_df) != len(ro_df):
        return rows
    cols = sorted(set(text_columns_for_df(en_df)) & set(text_columns_for_df(ro_df)))
    for col in cols:
        for idx in range(len(en_df)):
            en_text = str(en_df.iloc[idx][col])
            ro_text = str(ro_df.iloc[idx][col])
            en_words = simple_word_count(en_text)
            ro_words = simple_word_count(ro_text)
            rows.append({
                "dataset": dataset_name,
                "column": col,
                "row": idx,
                "en_chars": len(en_text),
                "ro_chars": len(ro_text),
                "en_words": en_words,
                "ro_words": ro_words,
                "char_ratio_ro_en": len(ro_text) / max(len(en_text), 1),
                "word_ratio_ro_en": ro_words / max(en_words, 1),
            })
    return rows

paired_rows = []
for name in sorted(set(en) & set(ro)):
    if name in ["test_a_labels", "test_b_labels"]:
        continue
    paired_rows.extend(paired_length_rows(name, en[name], ro[name]))

paired_lengths = pd.DataFrame(paired_rows)
paired_lengths.head()

In [ ]:
if not paired_lengths.empty:
    ratio_summary = paired_lengths.groupby(["dataset", "column"])[["char_ratio_ro_en", "word_ratio_ro_en"]].agg(
        ["count", "mean", "std", "median", "min", "max"]
    )
    display(ratio_summary)
    plot_hist(paired_lengths["word_ratio_ro_en"], "Romanian/English word-count ratio", "RO words / EN words", bins=50)

In [ ]:
if not paired_lengths.empty:
    plt.figure(figsize=(10, 4))
    if sns is not None:
        sns.boxplot(data=paired_lengths, x="dataset", y="word_ratio_ro_en")
    else:
        paired_lengths.boxplot(column="word_ratio_ro_en", by="dataset", rot=30)
    plt.axhline(1.0, color="gray", linestyle="--", linewidth=1)
    plt.title("Romanian/English word-count ratio by dataset")
    plt.xlabel("Dataset")
    plt.ylabel("RO words / EN words")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

## 12. Common Analysis: Translation Cache Quality

In [ ]:
cache_quality_rows = []
for en_text, ro_text in translation_cache.items():
    en_w = simple_word_count(en_text)
    ro_w = simple_word_count(ro_text)
    cache_quality_rows.append({
        "en_text": en_text,
        "ro_text": ro_text,
        "en_chars": len(en_text),
        "ro_chars": len(ro_text),
        "en_words": en_w,
        "ro_words": ro_w,
        "char_ratio_ro_en": len(ro_text) / max(len(en_text), 1),
        "word_ratio_ro_en": ro_w / max(en_w, 1),
        "empty_ro": len(ro_text.strip()) == 0,
        "identical_text": norm_text(en_text).lower() == norm_text(ro_text).lower(),
    })

cache_quality = pd.DataFrame(cache_quality_rows)
if not cache_quality.empty:
    display(cache_quality[["char_ratio_ro_en", "word_ratio_ro_en", "empty_ro", "identical_text"]].describe(include="all"))
    print("Empty Romanian translations:", int(cache_quality["empty_ro"].sum()))
    print("Identical EN/RO strings:", int(cache_quality["identical_text"].sum()))
else:
    print("No translation cache quality rows.")

In [ ]:
if not cache_quality.empty:
    suspicious = cache_quality[
        (cache_quality["empty_ro"]) |
        (cache_quality["identical_text"]) |
        (cache_quality["word_ratio_ro_en"] < 0.4) |
        (cache_quality["word_ratio_ro_en"] > 2.5)
    ].copy()
    print("Suspicious translation-cache entries:", len(suspicious))
    display(suspicious.sort_values("word_ratio_ro_en").head(20)[["en_text", "ro_text", "word_ratio_ro_en", "identical_text"]])

## 13. Common Analysis: Track B Label Coverage

In [ ]:
def track_b_label_coverage(test_b_df, labels_df, language, translation_cache=None):
    if test_b_df.empty or labels_df.empty or "text" not in test_b_df.columns:
        return pd.DataFrame()
    story_texts = set(test_b_df["text"].astype(str).map(norm_text))
    rows = []
    for col in ["anchor_text", "text_a", "text_b"]:
        if col not in labels_df.columns:
            continue
        vals = labels_df[col].astype(str).map(norm_text)
        if translation_cache is not None:
            vals = vals.map(lambda x: translation_cache.get(x, x))
        hits = vals.map(lambda x: x in story_texts)
        rows.append({
            "language": language,
            "label_column": col,
            "n": len(vals),
            "hits": int(hits.sum()),
            "missing": int((~hits).sum()),
            "hit_rate": float(hits.mean()) if len(hits) else 0.0,
        })
    return pd.DataFrame(rows)

tb_cov_en = track_b_label_coverage(en.get("test_b", pd.DataFrame()), en.get("test_b_labels", pd.DataFrame()), "English")
tb_cov_ro = track_b_label_coverage(ro.get("test_b", pd.DataFrame()), en.get("test_b_labels", pd.DataFrame()), "Romanian", translation_cache)
tb_cov = pd.concat([tb_cov_en, tb_cov_ro], ignore_index=True)
tb_cov

## 14. Qualitative Samples

In [ ]:
def show_parallel_samples(dataset_name="dev_a", n=3, seed=SEED):
    en_df = en.get(dataset_name, pd.DataFrame())
    ro_df = ro.get(dataset_name, pd.DataFrame())
    if en_df.empty or ro_df.empty or len(en_df) != len(ro_df):
        print("Cannot show aligned samples for", dataset_name)
        return
    cols = sorted(set(text_columns_for_df(en_df)) & set(text_columns_for_df(ro_df)))
    rng = np.random.default_rng(seed)
    idxs = rng.choice(len(en_df), size=min(n, len(en_df)), replace=False)
    for idx in idxs:
        print("=" * 100)
        print(f"Dataset={dataset_name} row={idx}")
        if "text_a_is_closer" in en_df.columns:
            print("Label text_a_is_closer:", bool(en_df.iloc[idx]["text_a_is_closer"]))
        for col in cols:
            print("-" * 100)
            print(f"{col} [EN]:", en_df.iloc[idx][col])
            print(f"{col} [RO]:", ro_df.iloc[idx][col])

show_parallel_samples("dev_a", n=2)

## 15. Tokenizer-Length Threshold Analysis for `max_len`

This section analyses how many tokens are required to represent the English stories under the tokenizer used by the baseline models.

In [27]:
OUT_DIR = Path(os.environ.get("OUT_DIR", "/kaggle/working"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
TOKENIZER_MODEL_NAME = os.environ.get("TOKENIZER_MODEL_NAME", "sentence-transformers/all-roberta-large-v1")
TOKENIZER_THRESHOLDS = [int(x) for x in os.environ.get("TOKENIZER_THRESHOLDS", "128,256,384,512").split(",")]

try:
    from transformers import AutoTokenizer
    tokenizer_for_lengths = AutoTokenizer.from_pretrained(TOKENIZER_MODEL_NAME)
    print("Loaded tokenizer:", TOKENIZER_MODEL_NAME)
except Exception as exc:
    tokenizer_for_lengths = None
    print("Tokenizer length analysis skipped because the tokenizer could not be loaded:", repr(exc))

def collect_unique_texts_for_tokenizer(data, dataset_names=None):
    rows = []
    dataset_names = dataset_names or list(data.keys())
    for dataset_name in dataset_names:
        df = data.get(dataset_name, pd.DataFrame())
        if df.empty:
            continue
        for col in text_columns_for_df(df):
            for text in df[col].dropna().astype(str):
                text_norm = norm_text(text)
                if text_norm:
                    rows.append({"dataset": dataset_name, "text_column": col, "text": text_norm})
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    return out.drop_duplicates(["dataset", "text"]).reset_index(drop=True)

def compute_token_lengths(texts, tokenizer):
    lengths = []
    for text in texts:
        encoded = tokenizer(str(text), add_special_tokens=True, truncation=False, return_attention_mask=False)
        lengths.append(len(encoded["input_ids"]))
    return lengths

tokenizer_dataset_names = [name for name in ["synthetic", "extra_synthetic", "extra_tell_me_again", "dev_a", "test_a", "test_b"] if name in en]
token_texts = collect_unique_texts_for_tokenizer(en, tokenizer_dataset_names)

if tokenizer_for_lengths is not None and not token_texts.empty:
    token_length_parts = []
    for dataset_name, group in token_texts.groupby("dataset", sort=False):
        tmp = group.copy()
        tmp["tokens"] = compute_token_lengths(tmp["text"].tolist(), tokenizer_for_lengths)
        token_length_parts.append(tmp)
    token_length_df = pd.concat(token_length_parts, ignore_index=True)
else:
    token_length_df = pd.DataFrame(columns=["dataset", "text_column", "text", "tokens"])

token_length_path = OUT_DIR / "eda_tokenizer_lengths.csv"
token_length_df.to_csv(token_length_path, index=False)
print("Saved tokenizer lengths:", token_length_path)
token_length_df.head()

In [ ]:
def summarize_token_thresholds(token_df, thresholds):
    rows = []
    for dataset_name, group in token_df.groupby("dataset", sort=False):
        tokens = group["tokens"].astype(float).to_numpy()
        if len(tokens) == 0:
            continue
        row = {
            "dataset": dataset_name,
            "n_unique_texts": int(len(tokens)),
            "mean_tokens": float(np.mean(tokens)),
            "median_tokens": float(np.median(tokens)),
            "p90_tokens": float(np.percentile(tokens, 90)),
            "p95_tokens": float(np.percentile(tokens, 95)),
            "p99_tokens": float(np.percentile(tokens, 99)),
            "max_tokens": float(np.max(tokens)),
        }
        for threshold in thresholds:
            row[f"pct_over_{threshold}"] = float((tokens > threshold).mean() * 100)
            row[f"mean_retained_at_{threshold}"] = float(np.mean(np.minimum(tokens, threshold) / np.maximum(tokens, 1)))
        rows.append(row)
    return pd.DataFrame(rows)

def make_threshold_long(token_df, thresholds):
    rows = []
    for dataset_name, group in token_df.groupby("dataset", sort=False):
        tokens = group["tokens"].astype(float).to_numpy()
        for threshold in thresholds:
            rows.append({
                "dataset": dataset_name,
                "max_len": threshold,
                "pct_truncated": float((tokens > threshold).mean() * 100) if len(tokens) else 0.0,
                "mean_retained_fraction": float(np.mean(np.minimum(tokens, threshold) / np.maximum(tokens, 1))) if len(tokens) else 0.0,
            })
    return pd.DataFrame(rows)

token_threshold_summary = summarize_token_thresholds(token_length_df, TOKENIZER_THRESHOLDS)
token_threshold_long = make_threshold_long(token_length_df, TOKENIZER_THRESHOLDS)

threshold_summary_path = OUT_DIR / "eda_tokenizer_threshold_summary.csv"
threshold_long_path = OUT_DIR / "eda_tokenizer_threshold_long.csv"
token_threshold_summary.to_csv(threshold_summary_path, index=False)
token_threshold_long.to_csv(threshold_long_path, index=False)

print("Saved threshold summary:", threshold_summary_path)
print("Saved threshold long table:", threshold_long_path)
token_threshold_summary

In [ ]:
if not token_threshold_long.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    if sns is not None:
        sns.lineplot(data=token_threshold_long, x="max_len", y="pct_truncated", hue="dataset", marker="o", ax=axes[0])
        sns.lineplot(data=token_threshold_long, x="max_len", y="mean_retained_fraction", hue="dataset", marker="o", ax=axes[1])
    else:
        for dataset_name, group in token_threshold_long.groupby("dataset"):
            axes[0].plot(group["max_len"], group["pct_truncated"], marker="o", label=dataset_name)
            axes[1].plot(group["max_len"], group["mean_retained_fraction"], marker="o", label=dataset_name)
    axes[0].set_title("Texts exceeding each max_len")
    axes[0].set_ylabel("Truncated texts (%)")
    axes[0].set_xlabel("max_len")
    axes[1].set_title("Mean retained token fraction")
    axes[1].set_ylabel("Retained fraction")
    axes[1].set_xlabel("max_len")
    for ax in axes:
        ax.grid(alpha=0.25)
        ax.legend(title="Dataset", bbox_to_anchor=(1.02, 1.0), loc="upper left")
    out_plot = OUT_DIR / "eda_tokenizer_max_len_thresholds.png"
    fig.savefig(out_plot, bbox_inches="tight")
    print("Saved plot:", out_plot)
else:
    print("No tokenizer threshold plot created.")

## 16. Organizer Synthetic vs Additional Data

In [ ]:
def triplet_text_columns(df):
    return [col for col in ["anchor_text", "text_a", "text_b"] if col in df.columns]

def triplet_key(row, cols):
    return " ||| ".join(norm_text(row.get(col, "")) for col in cols)

def synthetic_dataset_profile(df, source_name):
    if df.empty:
        return {}, pd.DataFrame()
    cols = triplet_text_columns(df)
    text_rows = []
    for role in cols:
        for text in df[role].dropna().astype(str):
            text_rows.append({
                "source": source_name,
                "role": role,
                "chars": len(text),
                "words": simple_word_count(text),
                "text": norm_text(text),
            })
    text_df = pd.DataFrame(text_rows)
    profile = {
        "source": source_name,
        "rows": int(len(df)),
        "text_columns": ", ".join(cols),
        "unique_story_texts": int(text_df["text"].nunique()) if not text_df.empty else 0,
        "mean_words_all_roles": float(text_df["words"].mean()) if not text_df.empty else 0.0,
        "median_words_all_roles": float(text_df["words"].median()) if not text_df.empty else 0.0,
        "max_words_all_roles": int(text_df["words"].max()) if not text_df.empty else 0,
    }
    if "text_a_is_closer" in df.columns:
        labels = df["text_a_is_closer"].astype(bool)
        profile["true_count"] = int(labels.sum())
        profile["false_count"] = int((~labels).sum())
        profile["true_pct"] = float(labels.mean() * 100)
    return profile, text_df

organizer_synth = en.get("synthetic", pd.DataFrame())
additional_synth = en.get("extra_generated", pd.DataFrame())
tell_me_again_synth = en.get("extra_tell_me_again", pd.DataFrame())

org_profile, org_texts = synthetic_dataset_profile(organizer_synth, "organizer_synthetic")
extra_profile, extra_texts = synthetic_dataset_profile(additional_synth, "additional_synthetic_new")
tellme_profile, tellme_texts = synthetic_dataset_profile(tell_me_again_synth, "additional_tell_me_again")

synthetic_profiles = pd.DataFrame([p for p in [org_profile, extra_profile, tellme_profile] if p])
synthetic_text_lengths = pd.concat([org_texts, extra_texts, tellme_texts], ignore_index=True) if not org_texts.empty or not extra_texts.empty or not tellme_texts.empty else pd.DataFrame()

# Calculate overlaps
cols_org = triplet_text_columns(organizer_synth)
cols_extra = triplet_text_columns(additional_synth)
cols_tellme = triplet_text_columns(tell_me_again_synth)

org_triplets = set(organizer_synth.apply(lambda r: triplet_key(r, cols_org), axis=1)) if not organizer_synth.empty else set()
extra_triplets = set(additional_synth.apply(lambda r: triplet_key(r, cols_extra), axis=1)) if not additional_synth.empty else set()
tellme_triplets = set(tell_me_again_synth.apply(lambda r: triplet_key(r, cols_tellme), axis=1)) if not tell_me_again_synth.empty else set()

org_story_texts = set(org_texts["text"]) if not org_texts.empty else set()
extra_story_texts = set(extra_texts["text"]) if not extra_texts.empty else set()
tellme_story_texts = set(tellme_texts["text"]) if not tellme_texts.empty else set()

synthetic_overlap = pd.DataFrame([{
    "organizer_rows": len(org_triplets),
    "additional_rows": len(extra_triplets),
    "tell_me_again_rows": len(tellme_triplets),
    "organizer_vs_additional_overlap": len(org_triplets & extra_triplets),
    "organizer_vs_tellme_overlap": len(org_triplets & tellme_triplets),
    "additional_vs_tellme_overlap": len(extra_triplets & tellme_triplets),
    "organizer_unique_story_texts": len(org_story_texts),
    "additional_unique_story_texts": len(extra_story_texts),
    "tellme_unique_story_texts": len(tellme_story_texts),
}])

synthetic_profiles_path = OUT_DIR / "eda_synthetic_profiles.csv"
synthetic_overlap_path = OUT_DIR / "eda_synthetic_overlap.csv"
synthetic_lengths_path = OUT_DIR / "eda_synthetic_role_lengths.csv"
synthetic_profiles.to_csv(synthetic_profiles_path, index=False)
synthetic_overlap.to_csv(synthetic_overlap_path, index=False)
synthetic_text_lengths.to_csv(synthetic_lengths_path, index=False)

print("Saved synthetic profiles:", synthetic_profiles_path)
print("Saved synthetic overlap:", synthetic_overlap_path)
print("Saved synthetic role lengths:", synthetic_lengths_path)
display(synthetic_profiles)
display(synthetic_overlap)

# Plot comparison
if not synthetic_text_lengths.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    if sns is not None:
        sns.boxplot(data=synthetic_text_lengths, x="role", y="words", hue="source", showfliers=False, ax=axes[0])
        sns.barplot(data=synthetic_text_lengths, x="role", y="words", hue="source", estimator=np.mean, ax=axes[1])
    else:
        synthetic_text_lengths.boxplot(column="words", by=["role", "source"], ax=axes[0], rot=45)
        synthetic_text_lengths.groupby(["role", "source"])["words"].mean().unstack().plot(kind="bar", ax=axes[1])
    axes[0].set_title("Synthetic text length distribution")
    axes[0].set_ylabel("Words")
    axes[0].set_xlabel("Triplet role")
    axes[1].set_title("Mean synthetic text length")
    axes[1].set_ylabel("Words")
    axes[1].set_xlabel("Triplet role")
    for ax in axes:
        ax.grid(axis="y", alpha=0.25)
        ax.legend(title="Source", bbox_to_anchor=(1.02, 1.0), loc="upper left")
    out_plot = OUT_DIR / "eda_synthetic_organizer_vs_additional_vs_tellme_lengths.png"
    fig.savefig(out_plot, bbox_inches="tight")
    print("Saved plot:", out_plot)
else:
    print("No synthetic length plot created.")

In [ ]:
if not synthetic_text_lengths.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    if sns is not None:
        sns.boxplot(data=synthetic_text_lengths, x="role", y="words", hue="source", showfliers=False, ax=axes[0])
        sns.barplot(data=synthetic_text_lengths, x="role", y="words", hue="source", estimator=np.mean, ax=axes[1])
    else:
        synthetic_text_lengths.boxplot(column="words", by=["role", "source"], ax=axes[0], rot=45)
        synthetic_text_lengths.groupby(["role", "source"])["words"].mean().unstack().plot(kind="bar", ax=axes[1])
    axes[0].set_title("Synthetic text length distribution")
    axes[0].set_ylabel("Words")
    axes[0].set_xlabel("Story role")
    axes[1].set_title("Mean synthetic text length")
    axes[1].set_ylabel("Words")
    axes[1].set_xlabel("Story role")
    for ax in axes:
        ax.grid(axis="y", alpha=0.25)
        ax.legend(title="Source", bbox_to_anchor=(1.02, 1.0), loc="upper left")
    out_plot = OUT_DIR / "eda_synthetic_organizer_vs_additional_lengths.png"
    fig.savefig(out_plot, bbox_inches="tight")
    print("Saved plot:", out_plot)
else:
    print("No synthetic length plot created.")

## 17. Summary Tables

In [ ]:
dataset_summary = shape_df.merge(
    labels_df[["language", "dataset", "true_pct"]] if not labels_df.empty else pd.DataFrame(columns=["language", "dataset", "true_pct"]),
    on=["language", "dataset"],
    how="left",
)
dataset_summary

In [ ]:
length_export = length_summary.pivot_table(
    index=["language", "dataset", "text_column"],
    columns="unit",
    values=["mean", "median", "max"],
    aggfunc="first",
)
length_export

In [ ]:
if not paired_lengths.empty:
    paired_export = paired_lengths.groupby(["dataset", "column"])[["char_ratio_ro_en", "word_ratio_ro_en"]].agg(["mean", "median", "min", "max"])
    display(paired_export)
else:
    print("No paired length export available.")